# **Сбор всех необходимых метрик и данных для дашборда с помощью CatBoost модели**

**Проект:** Анализ и визуализация данных с использованием Yandex DataLens: исследование по прогнозированию CTR

**Автор:** Грицан М.А., студент группы БПМИ-247, 2 курса

**Дата:** 26-03-2026  

**Цель:** Обучение CatBoost модели, примение модели на тестовой выборке для получения результатов в kaggle соревновании ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/overview), а также сбор всех необходимых для дашборда метрик

#### Необходимые библиотеки и настройка графиков:

In [1]:
%pip install catboost -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import zipfile

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.calibration import calibration_curve

In [3]:
%config InlineBackend.figure_format = 'retina'

sns.set(style='darkgrid', palette='deep')

plt.rcParams['figure.figsize'] = 8, 5
plt.rcParams['font.size'] = 12
plt.rcParams['savefig.format'] = 'pdf'

### 0. Загрука набора данных c kaggle
Скачиваем все файлы с соревнования ["Avazu Click-Through Rate Prediction"](https://www.kaggle.com/competitions/avazu-ctr-prediction/data) для дальнейшего использования.

In [ ]:
os.environ['KAGGLE_API_TOKEN'] = "ВАШ_KAGGLE_API_TOKEN"

print("✅ Kaggle API ключ установлен!")

✅ Kaggle API ключ установлен!


In [5]:
%pip install kaggle -q
!kaggle competitions download -c avazu-ctr-prediction -p ../

Note: you may need to restart the kernel to use updated packages.
avazu-ctr-prediction.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
with zipfile.ZipFile('../avazu-ctr-prediction.zip', 'r') as zip_ref:
    zip_ref.extractall('../avazu-ctr-prediction')

### 1. Считывание валидационной выборки и применение модели

In [4]:
def data_tranformer(df: pd.DataFrame):
    dt = pd.to_datetime(df['hour'], format='%y%m%d%H')
    df['day_of_week'] = dt.dt.dayofweek
    df['hour_of_day'] = dt.dt.hour

    return df

target = 'click'
categorical_features = [
    'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category',
    'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip',
    'device_model', 'device_type', 'device_conn_type',
    'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21',
    'day_of_week', 'hour_of_day'
]

In [5]:
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)

train_file = '../avazu-ctr-prediction/train.gz'
print(f"⏳ Загрузка валиадационной выборки из '{train_file}'...")

val_df = pd.read_csv(
    train_file,
    compression='gzip',
    skiprows=range(1, 30000001),
)
print(f"Прочитано {len(val_df)} строк, {len(val_df.columns)} колонки")

⏳ Загрузка валиадационной выборки из '../avazu-ctr-prediction/train.gz'...
Прочитано 10428967 строк, 24 колонки


In [6]:
os.makedirs(f'models', exist_ok=True)
model_path = 'models/catboost_ctr_model.cbm'

print("⏳ Достаем обученную модель...")
model = CatBoostClassifier()
model.load_model(model_path)

print(f"Количество признаков в модели: {len(model.feature_names_)}")

⏳ Достаем обученную модель...
Количество признаков в модели: 23


А теперь применим модель к валидационной выборке:

In [7]:
val_df = data_tranformer(val_df)
val_X = val_df[categorical_features]

predicted_target = "predicted_click"
predicted_ctr = "predicted_ctr"

val_df[predicted_ctr] = model.predict_proba(val_X)[:, 1]
val_df[predicted_target] = model.predict(val_X)

### 2. Сбор всех необходимых метрик и данных для дашборда

Решение отправленно, теперь надо подготовить и сохранить все данные для посторения 3-й вкладки дашборда в Yandex DataLens.

##### Базовые метрики на валидационной выборке:

In [11]:
print("⏳ Расчет ROC-AUC и LogLoss на валидационной выборке...")

roc_auc = roc_auc_score(val_df[target], val_df[predicted_ctr])
logloss = log_loss(val_df[target], val_df[predicted_ctr])

metrics_df = pd.DataFrame({
    'metric': ['ROC-AUC', 'Log Loss'],
    'value': [roc_auc, logloss]
})
metrics_df.to_csv('data/model_metrics.csv', index=False)
print("✅ Базовые метрики сохранены: data/model_metrics.csv")

metrics_df

⏳ Расчет ROC-AUC и LogLoss на валидационной выборке...
✅ Базовые метрики сохранены: data/model_metrics.csv


,metric,value
0,ROC-AUC,0.754833
1,Log Loss,0.383098


##### Данные для Lift:

In [12]:
print("⏳ Расcчет данных для Lift-таблицы...")

val_pool = Pool(val_df[categorical_features], cat_features=categorical_features)
all_shap_values = model.get_feature_importance(val_pool, type='ShapValues')

shap_values_only = all_shap_values[:, :-1]
shap_df = pd.DataFrame(shap_values_only, columns=categorical_features)[categorical_features]


total_traffic = len(val_df)
shap_lift_data = []

for category_name in categorical_features:
    temp_df = pd.DataFrame({
        'category_value': val_df[category_name].values,
        'shap_value': shap_df[category_name].values
    })

    grouped = temp_df.groupby('category_value').agg(
        traffic=('shap_value', 'count'),
        avg_shap=('shap_value', 'mean')
    ).reset_index()

    grouped['category_name'] = category_name
    grouped['traffic_share'] = (grouped['traffic'] / total_traffic)

    shap_lift_data.append(grouped[['category_name', 'category_value', 'avg_shap', 'traffic_share']])

shap_lift_df = pd.concat(shap_lift_data, ignore_index=True)

shap_lift_df.to_csv('data/shap_lift.csv', index=False)
print("✅ Lift-таблица на основе SHAP значений сохранена в data/shap_lift.csv")

shap_lift_df

⏳ Расcчет данных для Lift-таблицы...
✅ Lift-таблица на основе SHAP значений сохранена в data/shap_lift.csv


,category_name,category_value,avg_shap,traffic_share
0,C1,1001,-0.005442,0.000171
1,C1,1002,-0.001559,0.049125
2,C1,1005,-0.002124,0.927800
3,C1,1007,-0.006931,0.000516
4,C1,1008,0.002061,0.000039
...,...,...,...,...
2965616,hour_of_day,19,-0.047986,0.044178
2965617,hour_of_day,20,-0.048442,0.038150
2965618,hour_of_day,21,-0.047200,0.035881
2965619,hour_of_day,22,-0.040809,0.030282


##### Feature Importance:

In [13]:
print("⏳ Получение feature_importance обученной модели...")

feature_importance = model.get_feature_importance()
importance_df = pd.DataFrame({
    'feature': model.feature_names_,
    'importance': feature_importance
})

importance_df.to_csv('data/feature_importance.csv', index=False)
print("✅ Важность признаков сохранена в data/feature_importance.csv")

importance_df.sample(5)

⏳ Получение feature_importance обученной модели...
✅ Важность признаков сохранена в data/feature_importance.csv


,feature,importance
10,device_model,8.055427
2,site_id,14.808873
1,banner_pos,2.207459
8,device_id,8.813289
6,app_domain,1.349094


##### Топ связок признаков, которые выделил CatBoost:

In [14]:
print("⏳ Получение качества связок признаков обученной модели...")

interaction_importance = model.get_feature_importance(type="Interaction")

interactions_df = pd.DataFrame(
    interaction_importance,
    columns=['feature_1_idx', 'feature_2_idx', 'interaction_score']
)

feature_names = model.feature_names_
interactions_df['feature_1'] = interactions_df['feature_1_idx'].apply(lambda x: feature_names[int(x)])
interactions_df['feature_2'] = interactions_df['feature_2_idx'].apply(lambda x: feature_names[int(x)])
interactions_df = interactions_df[['feature_1', 'feature_2', 'interaction_score']]

interactions_df.to_csv('data/feature_interactions.csv', index=False)
print("✅ Cвязки признаков с качеством успешно сохранены в data/feature_interactions.csv")

interactions_df.sample(5)

⏳ Получение качества связок признаков обученной модели...
✅ Cвязки признаков с качеством успешно сохранены в data/feature_interactions.csv


,feature_1,feature_2,interaction_score
36,device_ip,C14,0.751403
123,C14,hour_of_day,0.181093
193,device_type,C17,0.075650
113,app_domain,device_conn_type,0.202350
47,app_id,C19,0.527225


##### Предсказанный CTR в сравнении с реальным в срезах разных фичей:

In [9]:
diff_ctr_data = []

needable_categorical_features = ['site_category', 'app_category', 'device_model', 'device_type', 'device_conn_type']

for category_name in needable_categorical_features:
    grouped = val_df.groupby(category_name).agg({target: 'mean', predicted_ctr: 'mean'}).reset_index()
    grouped.columns = ['category_value', 'real_ctr', 'predicted_ctr']
    grouped['category_name'] = category_name

    diff_ctr_data.append(grouped)

diff_ctr_df = pd.concat(diff_ctr_data, ignore_index=True)

diff_ctr_df.to_csv('data/diff_ctr.csv', index=False)
print("✅ Данные для графиков перепредсказонного и недопредсказанного CTR успешно сохранены в data/diff_ctr.csv")

diff_ctr_df.sample(5)

✅ Данные для графиков перепредсказонного и недопредсказанного CTR успешно сохранены в data/diff_ctr.csv


,category_value,real_ctr,predicted_ctr,category_name
5421,d4527c28,0.215945,0.213374,device_model
4817,be87996b,0.165929,0.201585,device_model
3934,9a7e2cd3,0.000000,0.030028,device_model
2682,69086f08,0.111111,0.198511,device_model
4707,ba04a9e3,0.130045,0.168676,device_model


##### Данные для диаграммы надежности модели:

In [16]:
print("⏳ Расчет данных для диаграммы надежности (Calibration Curve)...")

prob_true, prob_pred = calibration_curve(val_df[target], val_df[predicted_ctr], n_bins=10, strategy='uniform')

calibration_df = pd.DataFrame({
    'predicted_ctr': prob_pred,
    'real_ctr': prob_true
})
calibration_df['perfect_calibration'] = calibration_df['real_ctr']

calibration_df.to_csv('data/calibration_curve.csv', index=False)
print(f"\n✅ Данные для диаграммы надежности успешно сохранены в data/calibration_curve.csv")

calibration_df.sample(5)

⏳ Расчет данных для диаграммы надежности (Calibration Curve)...

✅ Данные для диаграммы надежности успешно сохранены в data/calibration_curve.csv


,predicted_ctr,real_ctr,perfect_calibration
4,0.437427,0.408656,0.408656
3,0.340708,0.309631,0.309631
8,0.831986,0.829654,0.829654
6,0.645920,0.662043,0.662043
0,0.050027,0.038761,0.038761
